# 🧠 Sage Ingestion Pipeline

"**What this does:** Chunks your preprocessed `.md` documents, writes ChromaDB + BM25 index, downloads a zip you drop straight into the project."

**Before you start:**

1. `Change runtime type to T4 GPU`
2. Upload your **`processed.zip`** and **`ingest.py`** when prompted
4. Run all cells top-to-bottom

In [1]:
#@title ✅ STEP 1: Verify T4 GPU is selected
import subprocess
import sys

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,driver_version,memory.total'],
    capture_output=True, text=True
    )
if result.returncode == 0:
    print("✅ GPU DETECTED:")
    gpu_available = True
else:
    print("❌ NO GPU FOUND")
    print(">>> Fix: Change runtime type. Then re-run this cell.")
    gpu_available = False
    raise SystemExit()

✅ GPU DETECTED:


In [2]:
#@title 📦 STEP 2: Install FastEmbed + dependencies

import subprocess
import sys
import os

def run_cmd(cmd, silent=False):
  """Run command with optional capture."""
  result = subprocess.run(cmd, shell=True, capture_output=silent, text=True)
  if not silent and result.returncode != 0:
    print(f"Error: {result.stderr}")
    raise RuntimeError(f"Command failed: {cmd}")
  return result

run_cmd(f"{sys.executable} -m pip uninstall -y -q onnxruntime onnxruntime-gpu", silent=True)
packages = [ "chromadb>=0.5.0", "rank-bm25>=0.2.2", "langchain-text-splitters>=0.0.1", "structlog>=24.1.0", "pyyaml>=6.0", "tqdm>=4.65.0",]
run_cmd(f"{sys.executable} -m pip install -q " + " ".join(packages), silent=True)
run_cmd(
    f"{sys.executable} -m pip install -q onnxruntime-gpu "
    f"--extra-index-url https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/onnxruntime-cuda-12/pypi/simple/",
    silent=False)
run_cmd(f"{sys.executable} -m pip install -q fastembed>=0.2.0", silent=True)

import onnxruntime as ort
providers = ort.get_available_providers()
print(f"\n✅ ONNX Runtime: {ort.__version__}")
print(f">>> Providers: {providers}")
cuda_provider = "CUDAExecutionProvider" in providers
print(f"\n>>> CUDA Provider: {'✅ AVAILABLE' if cuda_provider else '⚠️ CPU only (will still work, but slower)'}")


✅ ONNX Runtime: 1.26.0
>>> Providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']

>>> CUDA Provider: ✅ AVAILABLE


In [3]:
#@title 🧪 STEP 3: Test FastEmbed

import time
import torch
from fastembed import TextEmbedding
import onnxruntime as ort

print("Model: BAAI/bge-small-en-v1.5")
print("Dims: 384")

providers = (
    ["CUDAExecutionProvider", "CPUExecutionProvider"]
    if torch.cuda.is_available()
    else ["CPUExecutionProvider"]
)

print(f"Providers: {providers}")

t0 = time.time()
try:
    embedder = TextEmbedding(
        model_name="BAAI/bge-small-en-v1.5",
        cache_dir=".cache/fastembed",
        providers=providers,
        threads=8,
    )
    elapsed_load = time.time() - t0
    print(f"✅ Model loaded in {elapsed_load:.1f}s\n")
except Exception as e:
    print(f"❌ FastEmbed failed: {e}")
    import traceback
    traceback.print_exc()
    raise

print("=== Testing embedding generation ===")
test_texts = [
    "What is machine learning?",
    "Explain binary search algorithms",
    "How do databases work?",
]

t0 = time.time()
try:
    embeddings = list(embedder.embed(test_texts))
    elapsed_embed = time.time() - t0

    print(f"✅ Generated {len(embeddings)} embeddings in {elapsed_embed:.2f}s")
    print(f">>> Shape: {len(embeddings[0])} dims (384 expected)")
    print(f">>> Type: {type(embeddings[0])}")
    print(f">>> Speed: {len(test_texts) / elapsed_embed:.1f} docs/sec")
except Exception as e:
    print(f"❌ Embedding failed: {e}")
    import traceback
    traceback.print_exc()
    raise

print("\n✅ FastEmbed working perfectly!")

Model: BAAI/bge-small-en-v1.5
Dims: 384
Providers: ['CUDAExecutionProvider', 'CPUExecutionProvider']


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

✅ Model loaded in 5.8s

=== Testing embedding generation ===
✅ Generated 3 embeddings in 0.41s
>>> Shape: 384 dims (384 expected)
>>> Type: <class 'numpy.ndarray'>
>>> Speed: 7.2 docs/sec

✅ FastEmbed working perfectly!


In [5]:
#@title 📤 STEP 4: Upload your files
from google.colab import files
from pathlib import Path

print("Upload #1: processed.zip")

up1 = files.upload()
if not up1:
    raise SystemExit("No file uploaded.")
processed_zip = list(up1.keys())[0]
print(f"✅ Uploaded: {processed_zip}\n")

up2 = files.upload()
if not up2:
    raise SystemExit("No file uploaded.")
ingest_py = list(up2.keys())[0]
print(f"✅ Uploaded: {ingest_py}")

Upload #1: processed.zip


Saving processed.zip to processed.zip
✅ Uploaded: processed.zip



Saving ingest.py to ingest.py
✅ Uploaded: ingest.py


In [6]:
#@title 📂 STEP 5: Extract and prepare directories
import zipfile
from pathlib import Path
import shutil

WORK_DIR = Path("/content/sage_ingest")
OUTPUT_DIR = WORK_DIR / "outputs"
PROCESSED_ZIP = Path(processed_zip)

WORK_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Extracting {PROCESSED_ZIP.name}...")
with zipfile.ZipFile(PROCESSED_ZIP) as zf:
    zf.extractall(WORK_DIR)

PROC_DIR = WORK_DIR / "processed"
if not PROC_DIR.exists():
    md_files = list(WORK_DIR.rglob("*.md"))
    if md_files:
        PROC_DIR = md_files[0].parent
        while PROC_DIR != WORK_DIR and PROC_DIR.parent != WORK_DIR:
            PROC_DIR = PROC_DIR.parent
    else:
        raise FileNotFoundError(
            "No .md files in extracted zip!"
        )

md_count = len(list(PROC_DIR.rglob("*.md")))

if md_count == 0:
    raise FileNotFoundError("Found 0 .md files!")

print(f"✅ Found {md_count} .md files")
print(f">>> Location: {PROC_DIR}")
print(f">>> Output:   {OUTPUT_DIR}")

Extracting processed.zip...
✅ Found 20 .md files
>>> Location: /content/sage_ingest/processed
>>> Output:   /content/sage_ingest/outputs


In [7]:
#@title 🚀 STEP 6: Run ingestion
import os
import sys
import asyncio
import importlib.util
from pathlib import Path
import time
import subprocess

try:
    INGEST_PY_PATH = Path(list(up2.keys())[0]).resolve()
except (NameError, IndexError):
    INGEST_PY_PATH = Path("/content/ingest.py")

if not INGEST_PY_PATH.exists():
    raise FileNotFoundError(f"ingest.py not found at: {INGEST_PY_PATH}")
print(f"✅ ingest.py: {INGEST_PY_PATH}")

try:
    _proc_dir = PROC_DIR
    _out_dir = OUTPUT_DIR
except NameError:
    _proc_dir = Path("/content/sage_ingest/processed")
    _out_dir = Path("/content/sage_ingest/outputs")

if not _proc_dir.exists():
    _proc_dir = Path("/content/processed")
    if not _proc_dir.exists():
        raise FileNotFoundError(f"❌ Processed dir not found. Please re-run Step 5.")

_out_dir.mkdir(parents=True, exist_ok=True)
print(f"✅ Processed: {_proc_dir}")
print(f"✅ Output:    {_out_dir}\n")

print("📂 Loading ingest.py...")
try:
    spec = importlib.util.spec_from_file_location("ingest", INGEST_PY_PATH)
    ingest = importlib.util.module_from_spec(spec)
    sys.modules["ingest"] = ingest
    spec.loader.exec_module(ingest)
    print("ingest.py loaded successfully\n")
except Exception as e:
    print(f"Failed to load ingest.py: {e}")
    raise

def _embed_chunks_colab_safe(texts, embedder):
    import numpy as np
    return [v.astype(np.float16).tolist() for v in embedder.embed(texts, batch_size=512, parallel=1)]
ingest._embed_chunks = _embed_chunks_colab_safe

os.environ["SAGE_COLAB_MODE"] = "1"
ingest._configure_logging_fallback("info")

import nest_asyncio
nest_asyncio.apply()

print("===== STARTING INGESTION PIPELINE =====\n")
t_start = time.time()
try:
    loop = asyncio.get_event_loop()
    exit_code, courses_payload = loop.run_until_complete(
        ingest.run_pipeline(
            processed_root=_proc_dir,
            dry_run=False,
            force=True,
            course_filter=None,
            colab_mode=True,
            output_dir=_out_dir,
        )
    )
    print(f"\n✅ Pipeline completed in {time.time() - t_start:.1f}s")
    print(f"\n📚 Courses ingested: {courses_payload['courses']}")
except Exception as e:
    print(f"\n❌ Pipeline failed: {e}")
    raise

✅ ingest.py: /content/ingest.py
✅ Processed: /content/sage_ingest/processed
✅ Output:    /content/sage_ingest/outputs

📂 Loading ingest.py...
ingest.py loaded successfully

===== STARTING INGESTION PIPELINE =====

2026-06-01 12:14:57 [info     ] processed_dir_scanned          course_filter=None found=20 root=/content/sage_ingest/processed
2026-06-01 12:14:57 [info     ] ingest_pipeline_starting       dry_run=False force=True gpu=True total_docs=20
2026-06-01 12:14:58 [info     ] embedder_loaded                batch_size=512 gpu=True model=BAAI/bge-small-en-v1.5 providers=['CUDAExecutionProvider', 'CPUExecutionProvider']
2026-06-01 12:14:58 [info     ] global_embed_start             batch_size=512 gpu=True total_chunks=47353
2026-06-01 12:16:36 [info     ] global_embed_done              chunks=47353 chunks_per_sec=483 elapsed_ms=97994
2026-06-01 12:16:39 [info     ] chroma_collection_ready        collection=curriculum existing_chunks=0
2026-06-01 12:16:50 [info     ] document_ingested  

In [8]:
#@title 📊 STEP 7: Verify outputs
from pathlib import Path
import os

# Fallback for OUTPUT_DIR if variable is lost from memory
try:
    _out_dir = OUTPUT_DIR
except NameError:
    _out_dir = Path("/content/sage_ingest/outputs")

vectordb_dir = _out_dir / "vectordb"
bm25_file = _out_dir / "bm25_curriculum.pkl"

print(f"ChromaDB directory:  {vectordb_dir}")
if vectordb_dir.exists():
    files = list(vectordb_dir.rglob("*"))
    size_mb = sum(f.stat().st_size for f in files if f.is_file()) / 1e6
    print(f"Exists ({len(files)} files, {size_mb:.1f} MB)")
else:
    print(f"Missing!")

print(f"\nBM25 index file:     {bm25_file}")
if bm25_file.exists():
    size_mb = bm25_file.stat().st_size / 1e6
    print(f"Exists ({size_mb:.1f} MB)")
else:
    print(f"Missing!")

if _out_dir.exists():
    total_size = sum(f.stat().st_size for f in _out_dir.rglob('*') if f.is_file()) / 1e6
    print(f"\n📂 Total output size: {total_size:.1f} MB")
else:
    print("\nOutput directory not found.")
print(f"\ncourses.json:")
courses_json_file = _out_dir / "courses.json"
if courses_json_file.exists():
    import json as _json
    payload = _json.loads(courses_json_file.read_text())
    print(f"  ✅ Found: {payload}")
else:
    print(f"  ⚠️  Not written yet (will appear after full ingest run)")


ChromaDB directory:  /content/sage_ingest/outputs/vectordb
Exists (8 files, 338.6 MB)

BM25 index file:     /content/sage_ingest/outputs/bm25_curriculum.pkl
Exists (71.6 MB)

📂 Total output size: 410.3 MB

courses.json:
  ✅ Found: {'courses': []}


In [10]:
#@title 🔍 STEP 8: Testing

QUERY          = "What is supervised learning?"
TOP_K          = 5
COURSE_FILTER  = None

import json
import numpy as np
from pathlib import Path
from fastembed import TextEmbedding
import chromadb

try:
    _out_dir = OUTPUT_DIR
except NameError:
    _out_dir = Path("/content/sage_ingest/outputs")

VECTORDB_DIR   = _out_dir / "vectordb"
EMBED_MODEL    = "BAAI/bge-small-en-v1.5"
CACHE_DIR      = ".cache/fastembed"

import torch
_providers = (
    ["CUDAExecutionProvider", "CPUExecutionProvider"]
    if torch.cuda.is_available()
    else ["CPUExecutionProvider"]
)
try:
    _embedder
except NameError:
    print(f"Loading embedder ({EMBED_MODEL})…")
    _embedder = TextEmbedding(
        model_name=EMBED_MODEL,
        cache_dir=CACHE_DIR,
        providers=_providers,
    )
    print("✅ Embedder ready")

_q_vec = next(_embedder.embed([QUERY]))
_q_vec_fp16 = _q_vec.astype(np.float16).tolist()

_client     = chromadb.PersistentClient(path=str(VECTORDB_DIR))
_collection = _client.get_collection("curriculum")

_where = {"course_code": COURSE_FILTER} if COURSE_FILTER else None
_results = _collection.query(
    query_embeddings=[_q_vec_fp16],
    n_results=TOP_K,
    where=_where,
    include=["documents", "metadatas", "distances"],
)

docs      = _results["documents"][0]
metas     = _results["metadatas"][0]
distances = _results["distances"][0]

print(f"\n{'═'*72}")
print(f"  Query : {QUERY}")
print(f"  Filter: {COURSE_FILTER or 'all courses'}  |  Top-{TOP_K} chunks")
print(f"{'═'*72}\n")

for rank, (doc, meta, dist) in enumerate(zip(docs, metas, distances), 1):
    similarity = 1 - dist
    course     = meta.get("course_code", "?")
    title      = meta.get("doc_title",   "?")
    chunk_idx  = meta.get("chunk_index", "?")
    preview    = doc[:300].replace("\n", " ")
    print(f"[{rank}] sim={similarity:.4f}  course={course}  doc={title}  chunk={chunk_idx}")
    print(f"    {preview}")
    print()
_courses_file = _out_dir / "courses.json"
if _courses_file.exists():
    _courses = json.loads(_courses_file.read_text())["courses"]
    print(f"Available courses for filtering: {_courses}")



════════════════════════════════════════════════════════════════════════
  Query : What is supervised learning?
  Filter: all courses  |  Top-5 chunks
════════════════════════════════════════════════════════════════════════

[1] sim=0.8003  course=  doc=Artificial Intelligence A Modern Approach (3rd Ed) Stuart Russell  chunk=3882
 where each yj was generated by an unkn

[2] sim=0.7497  course=  doc=Artificial Intelligence A Modern Approach (3rd Ed) Stuart Russell  chunk=3879
 In supervised learning the agent observes some example input-output pairs and l

[3] sim=0.7422  course=  doc=Artificial Intelligence A Modern Approach (3rd Ed) Stuart Russell  chunk=4204
 • Inductive learning invol

[4] sim=0.7347  course=  doc=Artificial Intelligence A Modern Approach (3rd Ed) Stuart Russell  chunk=5439
 the robot to drive fa

[5] sim=0.7226  course=  doc=Artificial Intelligence A Modern Approach (3rd Ed) Stuart Russell  chunk=3881
 supervised learning. But in reality some of the

Available cou

In [ ]:
#@title 📥 STEP 9: Download outputs
import zipfile
from pathlib import Path
from google.colab import files as colab_files

# Fallback for OUTPUT_DIR if variable is lost from memory
try:
    _out_dir = OUTPUT_DIR
except NameError:
    _out_dir = Path("/content/sage_ingest/outputs")

OUTPUT_ZIP = Path("/content/sage_outputs.zip")

if not _out_dir.exists():
    raise FileNotFoundError(f"Output directory not found at {_out_dir}. Please run the ingestion step first.")


with zipfile.ZipFile(OUTPUT_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in _out_dir.rglob("*"):
        if f.is_file():
            arcname = f.relative_to(_out_dir)
            zf.write(f, arcname)
            size_kb = f.stat().st_size / 1e3
            print(f"{arcname} ({size_kb:.0f} KB)")

if OUTPUT_ZIP.exists():
    size_mb = OUTPUT_ZIP.stat().st_size / 1e6
    print(f"\n✅ Total package: {size_mb:.1f} MB")
    colab_files.download(str(OUTPUT_ZIP))
    print(f"✅ Downloaded: sage_outputs.zip")
else:
    print("❌ Failed to create zip file.")

bm25_curriculum.pkl (177653 KB)
vectordb/ingest_cache.json (8 KB)
vectordb/chroma.sqlite3 (633381 KB)
vectordb/6bcaae45-ac00-4a2f-a794-1d87b28c2164/length.bin (502 KB)
vectordb/6bcaae45-ac00-4a2f-a794-1d87b28c2164/header.bin (0 KB)
vectordb/6bcaae45-ac00-4a2f-a794-1d87b28c2164/index_metadata.pickle (23669 KB)


KeyboardInterrupt: 